# 02 — Data Cleaning & ETL Pipeline

**Project:** Online Retail Revenue Intelligence  
**Team:** Hopper_DataDrift_OnlineRetailAnalytics | Newton School of Technology  

**Objective:**  
Execute the full cleaning pipeline using modular functions from `scripts/etl_pipeline.py`. Every transformation step is logged and justified. The output is a fully standardized, analysis-ready dataset committed to `data/processed/`.

**Cleaning Steps:**
1. Column normalization (snake_case)
2. Duplicate removal
3. Date parsing + temporal feature engineering
4. Cancellation flagging
5. Invalid quantity/price removal
6. Missing Customer ID handling
7. Text standardization (Description, Country)
8. Revenue derivation
9. Product category derivation

## 2.1 — Environment Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.etl_pipeline import (
    normalize_columns, remove_duplicates, parse_dates,
    flag_cancellations, remove_invalid_transactions,
    handle_missing_customers, standardize_text,
    derive_revenue, derive_category, run_full_pipeline, save_processed
)

RAW_PATH       = PROJECT_ROOT / 'data' / 'raw' / 'online_retail_II_raw.csv'
PROCESSED_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_retail.csv'

print('Setup complete. ETL functions imported.')

Setup complete. ETL functions imported.


## 2.2 — Load Raw Data

In [2]:
try:
    raw_df = pd.read_csv(RAW_PATH, encoding='ISO-8859-1')
except UnicodeDecodeError:
    raw_df = pd.read_csv(RAW_PATH, encoding='latin-1')

print(f'Raw shape: {raw_df.shape}')
raw_df.head(3)

Raw shape: (1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom


## 2.3 — Step-by-Step Pipeline Execution

Each step prints a log line explaining what changed and why.

In [3]:
# STEP 1 — Normalize column names to snake_case
# Rationale: consistent column naming prevents KeyError bugs across notebooks
df = normalize_columns(raw_df)
print('Columns after normalization:', list(df.columns))

INFO | Step 1 | Columns normalized: ['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']


Columns after normalization: ['invoice', 'stockcode', 'description', 'quantity', 'invoicedate', 'price', 'customer_id', 'country']


In [4]:
# STEP 2 — Remove exact duplicate rows
# Rationale: exact duplicates inflate counts and distort KPIs
before = len(df)
df = remove_duplicates(df)
print(f'Rows before: {before:,} | After: {len(df):,} | Removed: {before - len(df):,}')

INFO | Step 2 | Duplicates removed: 34335 rows dropped


Rows before: 1,067,371 | After: 1,033,036 | Removed: 34,335


In [5]:
# STEP 3 — Parse InvoiceDate and derive temporal features
# Rationale: raw date is a string; we need year/month/quarter/day for time-series analysis
df = parse_dates(df, date_col='invoicedate')
print('Date range:', df['invoicedate'].min(), '→', df['invoicedate'].max())
print('Derived columns added: year, month, month_name, quarter, day_of_week, hour')
df[['invoicedate','year','month','month_name','quarter','day_of_week','hour']].head(3)

INFO | Step 3 | Dates parsed. Null dates: 0


Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
Derived columns added: year, month, month_name, quarter, day_of_week, hour


,invoicedate,year,month,month_name,quarter,day_of_week,hour
0,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7
1,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7
2,2009-12-01 07:45:00,2009,12,December,4,Tuesday,7


In [6]:
# STEP 4 — Flag cancellations
# Rationale: cancellations (Invoice starting with 'C') are real business events.
# We flag rather than drop so cancellation rate KPI can be computed.
df = flag_cancellations(df, invoice_col='invoice')
total = len(df)
cancelled = df['is_cancelled'].sum()
print(f'Total rows: {total:,} | Cancelled: {cancelled:,} ({cancelled/total*100:.1f}%)')

INFO | Step 4 | Cancellations flagged: 19104 rows


Total rows: 1,033,036 | Cancelled: 19,104 (1.8%)


In [7]:
# STEP 5 — Remove invalid transactions (non-cancelled rows with qty<=0 or price<=0)
# Rationale: these are data errors — a product cannot have zero or negative price
# unless it is a cancellation/refund (already flagged above).
before = len(df)
df = remove_invalid_transactions(df)
print(f'Rows before: {before:,} | After: {len(df):,} | Removed: {before - len(df):,}')

INFO | Step 5 | Invalid transactions removed: 6019 rows


Rows before: 1,033,036 | After: 1,027,017 | Removed: 6,019


In [8]:
# STEP 6 — Handle missing Customer IDs
# Rationale: ~25% of rows lack Customer ID (guest checkouts).
# Dropping them would lose significant revenue signal.
# Strategy: fill with 'GUEST' — included in revenue KPIs, excluded from RFM.
before_null = df['customer_id'].isna().sum()
df = handle_missing_customers(df, customer_col='customer_id')
print(f'Missing Customer IDs filled: {before_null:,}')
print('Guest rows:', (df['customer_id'] == 'GUEST').sum())

INFO | Step 6 | Missing Customer IDs filled with GUEST: 229202 rows


Missing Customer IDs filled: 229,202
Guest rows: 229202


In [9]:
# STEP 7 — Standardize text: Description and Country
# Rationale: inconsistent casing and whitespace create duplicate categories in Tableau
# e.g. 'UNITED KINGDOM' and 'United Kingdom' would appear as two separate countries.
before = len(df)
df = standardize_text(df)
print(f'Rows after noise removal: {len(df):,} (removed {before - len(df):,} noise rows)')
print('Sample descriptions:', df['description'].dropna().unique()[:5])
print('Sample countries:', df['country'].unique()[:8])

INFO | Step 7 | Text columns standardized.


Rows after noise removal: 1,027,017 (removed 0 noise rows)
Sample descriptions: <StringArray>
['15Cm Christmas Glass Ball 20 Lights',                  'Pink Cherry Lights',
                 'White Cherry Lights',         'Record Frame 7" Single Size',
      'Strawberry Ceramic Trinket Box']
Length: 5, dtype: str
Sample countries: <StringArray>
['United Kingdom',         'France',            'Usa',        'Belgium',
      'Australia',           'Eire',        'Germany',       'Portugal']
Length: 8, dtype: str


In [10]:
# STEP 8 — Derive Revenue = Quantity × Price
# Rationale: revenue is the primary business KPI; it must be computed from source columns
# so the derivation is fully reproducible and auditable.
df = derive_revenue(df)
gross_rev = df.loc[~df['is_cancelled'], 'revenue'].sum()
print(f'Total gross revenue (non-cancelled): £{gross_rev:,.2f}')
df[['invoice','description','quantity','price','revenue','is_cancelled']].head(5)

INFO | Step 8 | Revenue derived. Total gross revenue: £20,476,260.45


Total gross revenue (non-cancelled): £20,476,260.45


,invoice,description,quantity,price,revenue,is_cancelled
0,489434,15Cm Christmas Glass Ball 20 Lights,12,6.95,83.4,False
1,489434,Pink Cherry Lights,12,6.75,81.0,False
2,489434,White Cherry Lights,12,6.75,81.0,False
3,489434,"Record Frame 7"" Single Size",48,2.10,100.8,False
4,489434,Strawberry Ceramic Trinket Box,24,1.25,30.0,False


In [11]:
# STEP 9 — Derive product category from first word of Description
# Rationale: the dataset lacks an explicit category column.
# The first keyword of the Description (e.g. 'White', 'Red', 'Vintage') provides
# a useful grouping dimension for Tableau filters.
df = derive_category(df)
print(f'Unique categories: {df["category"].nunique()}')
print('Top 10 categories:')
print(df['category'].value_counts().head(10))

INFO | Step 9 | Category column derived. Unique categories: 975


Unique categories: 975
Top 10 categories:
category
Set        51269
Red        41748
Jumbo      31890
Pack       27657
Pink       25222
Lunch      23922
Vintage    19788
White      17667
Blue       16624
Wooden     13365
Name: count, dtype: int64


## 2.4 — Cleaning Summary

In [12]:
print('=== CLEANING SUMMARY ===')
print(f'  Raw rows       : {len(raw_df):,}')
print(f'  Cleaned rows   : {len(df):,}')
print(f'  Rows removed   : {len(raw_df) - len(df):,} ({(len(raw_df)-len(df))/len(raw_df)*100:.1f}%)')
print(f'  Final columns  : {len(df.columns)}')
print(f'  Cancelled rows : {df["is_cancelled"].sum():,}')
print(f'  GUEST customers: {(df["customer_id"]=="GUEST").sum():,}')
print()
print('Final column list:')
for col in df.columns:
    print(f'  {col} — {df[col].dtype}')

=== CLEANING SUMMARY ===
  Raw rows       : 1,067,371
  Cleaned rows   : 1,027,017
  Rows removed   : 40,354 (3.8%)
  Final columns  : 18
  Cancelled rows : 19,104
  GUEST customers: 229,202

Final column list:
  invoice — str
  stockcode — str
  description — str
  quantity — int64
  invoicedate — datetime64[us]
  price — float64
  customer_id — str
  country — str
  year — int32
  month — int32
  month_name — str
  quarter — int32
  day_of_week — str
  hour — int32
  is_cancelled — bool
  revenue — float64
  abs_revenue — float64
  category — object


In [13]:
# Confirm zero remaining nulls in critical columns
critical_cols = ['invoice', 'description', 'quantity', 'price', 'invoicedate', 'country', 'revenue']
null_check = df[critical_cols].isna().sum()
print('Null check on critical columns (should all be 0):')
print(null_check.to_string())

Null check on critical columns (should all be 0):
invoice        0
description    0
quantity       0
price          0
invoicedate    0
country        0
revenue        0


## 2.5 — Save Cleaned Dataset

In [14]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Cleaned dataset saved to: {PROCESSED_PATH}')
print(f'Shape: {df.shape}')
df.head(3)

Cleaned dataset saved to: /Users/rudranshgupta/Hopper_DataDrift_OnlineRetailAnalytics/data/processed/cleaned_retail.csv
Shape: (1027017, 18)


,invoice,stockcode,description,quantity,invoicedate,price,customer_id,country,year,month,month_name,quarter,day_of_week,hour,is_cancelled,revenue,abs_revenue,category
0,489434,85048,15Cm Christmas Glass Ball 20 Lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,2009,12,December,4,Tuesday,7,False,83.4,83.4,15Cm
1,489434,79323P,Pink Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009,12,December,4,Tuesday,7,False,81.0,81.0,Pink
2,489434,79323W,White Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,2009,12,December,4,Tuesday,7,False,81.0,81.0,White
